<div dir="rtl">

# بررسی مستقل اتصال API

این نوت‌بوک از داده‌ها و سل‌های پروژه مستقل است. کلید و آدرس خصوصی چاپ یا داخل فایل ذخیره نمی‌شوند.

</div>


In [1]:
# API-DIAG-C01 — Configuration inspection, no network
import os
import json
import hashlib
import time
import urllib.request
import urllib.error
import urllib.parse

diag_url = os.environ.get("ARVAN_CHAT_URL", "").strip()
diag_model = os.environ.get("ARVAN_CHAT_MODEL", "").strip()
diag_keys = {
    os.environ.get(name, "").strip()
    for name in ("ARVAN_CHAT_API_KEY", "ARVAN_AI_API_KEY")
} - {""}

diag_parts = urllib.parse.urlsplit(diag_url)
diag_config_valid = (
    diag_parts.scheme == "https"
    and bool(diag_parts.hostname)
    and not diag_parts.username
    and not diag_parts.password
    and not diag_parts.query
    and not diag_parts.fragment
    and bool(diag_model)
    and len(diag_keys) == 1
)

print(json.dumps({
    "status": "CONFIG_READY" if diag_config_valid else "CONFIG_MISSING_OR_INVALID",
    "url_present": bool(diag_url),
    "model": diag_model or None,
    "distinct_nonempty_keys": len(diag_keys),
    "https": diag_parts.scheme == "https",
    "api_calls": 0,
    "secret_values_printed": False,
}, ensure_ascii=False, indent=2))


{
  "status": "CONFIG_READY",
  "url_present": true,
  "model": "Qwen3-30B-A3B",
  "distinct_nonempty_keys": 1,
  "https": true,
  "api_calls": 0,
  "secret_values_printed": false
}


<div dir="rtl">

ابتدا سل تنظیمات را اجرا کنید. اگر CONFIG_READY بود، در سل بعد RUN_DIRECT_TEST را True کنید و فقط یک بار اجرا کنید. پس از تست آن را به False برگردانید. اجرای مجدد با True یک درخواست تازه می‌فرستد.

</div>


In [2]:
# API-DIAG-C02 — One direct HTTP chat request, no SDK, no retries
RUN_DIRECT_TEST = True

if not RUN_DIRECT_TEST:
    print(json.dumps({"status": "DIRECT_TEST_DISABLED", "api_calls": 0}))
else:
    if not diag_config_valid:
        raise ValueError("Complete the environment configuration first.")

    # ARVAN_CHAT_URL is the same base URL used by the OpenAI client.
    diag_endpoint = diag_url.rstrip("/") + "/chat/completions"
    diag_body = json.dumps({
        "model": diag_model,
        "messages": [
            {"role": "system", "content": 'Reply only with {"connection":"ok"}.'},
            {"role": "user", "content": "Connection check."},
        ],
        "max_tokens": 1024,
    }, ensure_ascii=False).encode("utf-8")

    diag_request = urllib.request.Request(
        diag_endpoint,
        data=diag_body,
        headers={
            "Authorization": "Bearer " + next(iter(diag_keys)),
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )

    # Prevent redirects from forwarding credentials to a different endpoint.
    class DiagNoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    diag_opener = urllib.request.build_opener(DiagNoRedirect())
    diag_report = {
        "status": "DIRECT_HTTP_TEST_FAILED",
        "api_request_attempts": 1,
        "automatic_retries": 0,
        "http_status": None,
        "error_type": None,
        "diagnostic_headers": {},
        "expected_json_received": False,
        "response_model": None,
        "usage": None,
        "project_data_used": False,
    }
    diag_started = time.monotonic()

    try:
        with diag_opener.open(diag_request, timeout=45) as response:
            diag_raw = response.read()
            diag_report["http_status"] = response.status
            diag_report["diagnostic_headers"] = {
                name: response.headers[name]
                for name in ("x-request-id", "date", "server")
                if name in response.headers
            }

        diag_payload = json.loads(diag_raw)
        diag_choice = (diag_payload.get("choices") or [{}])[0]
        diag_content = (diag_choice.get("message") or {}).get("content")
        diag_report["response_model"] = diag_payload.get("model")
        diag_report["usage"] = diag_payload.get("usage")
        diag_report["finish_reason"] = diag_choice.get("finish_reason")

        try:
            diag_report["expected_json_received"] = (
                json.loads(diag_content) == {"connection": "ok"}
            )
        except (TypeError, ValueError):
            pass

        diag_report["model_name_tail_matches"] = (
            str(diag_payload.get("model") or "").split("/")[-1].casefold()
            == diag_model.split("/")[-1].casefold()
        )
        if (
            diag_report["http_status"] == 200
            and diag_report["expected_json_received"]
            and diag_report["model_name_tail_matches"]
            and diag_report["finish_reason"] == "stop"
        ):
            diag_report["status"] = "DIRECT_HTTP_SMALL_CHAT_PASS"
        else:
            diag_report["status"] = "DIRECT_HTTP_RESPONSE_REQUIRES_INSPECTION"

    except urllib.error.HTTPError as error:
        diag_error_body = error.read()
        diag_report.update({
            "http_status": error.code,
            "error_type": type(error).__name__,
            "error_body_bytes": len(diag_error_body),
            "error_body_sha256": hashlib.sha256(diag_error_body).hexdigest(),
            "diagnostic_headers": {
                name: error.headers[name]
                for name in ("x-request-id", "date", "server")
                if name in error.headers
            },
        })
    except Exception as error:
        diag_report["error_type"] = type(error).__name__

    diag_report["elapsed_seconds"] = round(
        time.monotonic() - diag_started, 2
    )
    print(json.dumps(diag_report, ensure_ascii=False, indent=2))


{
  "status": "DIRECT_HTTP_TEST_FAILED",
  "api_request_attempts": 1,
  "automatic_retries": 0,
  "http_status": 502,
  "error_type": "HTTPError",
  "diagnostic_headers": {
    "x-request-id": "39a8d8a2da60802cc253a93cd4b4fc35",
    "date": "Tue, 06 Oct 2026 15:48:27 GMT",
    "server": "ArvanCloud"
  },
  "expected_json_received": false,
  "response_model": null,
  "usage": null,
  "project_data_used": false,
  "error_body_bytes": 140,
  "error_body_sha256": "9dc38cea5772e62fb54a88132a65af6fd475bd22addb43d6e5d62ceb38959c4d",
  "elapsed_seconds": 7.78
}
